# 01 — Mark 1 — Probability, Contrast & Localization Diagnostic

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **01 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_1_probability_contrast_localization_diagnostic.ipynb`

## Objective

Does the frozen epoch-8 multi-task checkpoint localize tumour signal anywhere in the probability map, and can global calibration (tumor threshold + predicted-liver support) recover it into a usable prediction?

## Inputs (read-only)

- `Practice/multitask_liver_tumor_outputs/multitask_best.pth` (epoch-8, SHA-256 verified)
- Verified manifest `build_corrected_20260713_214847_v2` (val split, 13 volumes / 10,685 slices)
- Frozen probability cache `mark 1/mark_1_outputs/probability_cache/` (REUSE) or live inference (REBUILD)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_1_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_1_gate_result.json` |
| `cache_coverage.csv` |
| `probability_slice_statistics.csv` |
| `calibration_configuration_results.csv` |
| `calibration_patient_metrics.csv` |
| `hu_contrast_per_slice.csv` |
| `hu_contrast_per_volume.csv` |
| `bootstrap_confidence_intervals.csv` |
| `probability_cache/ (13 × volume_*.npz)` |
| `probability_localization_104.png` |
| `probability_localization_116.png` |
| `probability_population_histograms.png` |
| `calibration_frontier_dashboard.png` |

**Visualizations produced by this notebook:** `probability_localization_104.png`, `probability_localization_116.png`, `probability_population_histograms.png`, `calibration_frontier_dashboard.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: Practice/multitask_liver_tumor_outputs/multitask_best.pth, build_corrected_20260713_214847_v2, mark 1/mark_1_outputs/probability_cache/"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_1_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Calibration gate FAILED.** No global configuration passed every guardrail. Best raw config (tumor threshold 0.70): mean patient Dice ≈ 0.333, V104 ≈ 0, V116 ≈ 0 — the signal is present but **mislocalized/suppressed**, not merely under-thresholded → next experiment: predicted-liver ROI.

## Gate

`mark_1_gate_result.json` — `calibration_gate_passed` expected **False** (failure category: mislocalized signal)

## Run notes

Validation-only, no training. Test split locked. 1.9 runs a 1,000-iteration bootstrap and compares the recomputed gate to `mark 1/mark_1_outputs/mark_1_gate_result.json` (drift < 1e-4).

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_1"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 1 — Mark 1: Probability, Contrast & Localization Diagnostic

**Original:** `mark 1/mark_1_probability_contrast_localization_diagnostic.ipynb`

## Question

Does the frozen epoch-8 multi-task checkpoint already localize tumour signal somewhere in the
probability map, and can global calibration (threshold + predicted-liver support) recover it?

## Key finding (reproduced)

- **Calibration gate: FAILED.** No global configuration passed every guardrail.
- Best observed configuration (raw mode, tumor threshold 0.70): mean patient Dice ≈ 0.333, V104 ≈ 0,
  V116 ≈ 0 — the signal is present but **mis-localized / suppressed**, not merely under-thresholded.
- Classification: `mislocalized_or_absent_signal` → next experiment = predicted-liver ROI.

## Contract

- Validation-only. No training. No patient-specific thresholds. Test split locked.
- Guardrail targets (Mark 1): mean patient Dice ≥ 0.406915, V104 ≥ 0.50, V116 ≥ 0.05,
  Q1 detection ≥ 45%, positive predicted-empty ≤ 20%, empty-slice FP ≤ 15%.

### 1.1 Load the epoch-8 checkpoint strictly and verify geometry

In [2]:
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

checkpoint = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 8
assert checkpoint["manifest_sha256"] == EXPECTED_MANIFEST_SHA256

model = MobileNetV2UNet(in_channels=1, out_channels=2, pretrained=False)
model.load_state_dict(checkpoint["model_state"], strict=True)
model.to(DEVICE).eval()

with torch.inference_mode():
    probe = model(torch.zeros(1, 1, 256, 256, device=DEVICE))
    probe_prob = torch.sigmoid(probe)
assert tuple(probe.shape) == (1, 2, 256, 256)
assert torch.isfinite(probe_prob).all()
print("PASS: strict epoch-8 checkpoint load;"
      f" checkpoint={source_checkpoint_hash[:12]}...")

PASS: strict epoch-8 checkpoint load; checkpoint=9c4160bbd688...


### 1.2 Reproduce validation preprocessing and build the validation loader

In [3]:
class Mark1ValidationDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.rows = base_dataset.rows

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        sample = self.base[index]
        image = image_robust_normalize(sample["image"][0].numpy())
        with Image.open(self.rows[index]["organ_mask_path"]) as handle:
            organ = (np.asarray(handle.convert("L"), dtype=np.uint8) > 0).astype(np.uint8)
        return {
            "image": torch.from_numpy(image[None]).float(),
            "tumor_mask": sample["mask"].to(torch.uint8),
            "organ_mask": torch.from_numpy(organ[None]),
            "sample_id": sample["sample_id"],
            "volume_id": int(sample["volume_id"]),
            "slice_index": int(sample["slice_index"]),
        }

validation_base = VerifiedManifestDataset(
    MANIFEST_PATH, split="val", root_dir=DATASET_ROOT,
    target="tumor", transform=None, validate_paths=True,
)
validation_dataset = Mark1ValidationDataset(validation_base)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
assert len(validation_dataset) == 10_685
print(f"READY: validation loader contains {len(validation_dataset):,} slices.")

READY: validation loader contains 10,685 slices.


### 1.3 Cache full-image probabilities per volume (reuse frozen cache or rebuild)

In [4]:
ORIG_CACHE = MARK1_DIR / "mark_1_outputs" / "probability_cache"
CACHE_DIR = OUT_CACHE["mark_1"]
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def save_volume_cache(volume_id, bucket, target_dir):
    order = np.argsort(bucket["slice_index"])
    payload = {key: np.asarray(value)[order] for key, value in bucket.items()}
    np.savez_compressed(target_dir / f"volume_{volume_id}.npz", **payload)


def build_probability_cache():
    current_volume, bucket, processed = None, None, []
    model.eval()
    with torch.inference_mode():
        for batch in validation_loader:
            probabilities = torch.sigmoid(
                model(batch["image"].to(DEVICE, non_blocking=True))).cpu().numpy()
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                if current_volume is None or volume_id != current_volume:
                    if current_volume is not None:
                        save_volume_cache(current_volume, bucket, CACHE_DIR)
                        processed.append(current_volume)
                    current_volume = volume_id
                    bucket = {"sample_id": [], "slice_index": [],
                              "tumor_truth": [], "organ_truth": [],
                              "liver_probability": [], "tumor_probability": []}
                bucket["sample_id"].append(str(sample_id))
                bucket["slice_index"].append(int(batch["slice_index"][index]))
                bucket["tumor_truth"].append(batch["tumor_mask"][index, 0].numpy().astype(np.uint8))
                bucket["organ_truth"].append(batch["organ_mask"][index, 0].numpy().astype(np.uint8))
                bucket["liver_probability"].append(probabilities[index, 0].astype(np.float16))
                bucket["tumor_probability"].append(probabilities[index, 1].astype(np.float16))
    if current_volume is not None:
        save_volume_cache(current_volume, bucket, CACHE_DIR)
        processed.append(current_volume)
    return processed


existing = sorted(CACHE_DIR.glob("volume_*.npz"))
if REUSE_CACHES and len(existing) == 13:
    print(f"REUSE: {len(existing)} patient caches already present in {CACHE_DIR}.")
elif REUSE_CACHES and len(list(ORIG_CACHE.glob("volume_*.npz"))) == 13:
    import shutil
    for p in ORIG_CACHE.glob("volume_*.npz"):
        shutil.copy2(p, CACHE_DIR / p.name)
    print("REUSE: copied 13 frozen patient caches from mark_1_outputs/probability_cache.")
else:
    processed = build_probability_cache()
    print(f"REBUILD: cached {len(processed)} volumes.")

m1_cache = {}
coverage = []
for path in sorted(CACHE_DIR.glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        item = {key: payload[key] for key in payload.files}
    expected = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)].sort_values("slice_index")
    assert len(item["sample_id"]) == len(expected)
    assert item["sample_id"].astype(str).tolist() == expected["sample_id"].astype(str).tolist()
    assert np.array_equal(item["slice_index"], expected["slice_index"].to_numpy())
    for key in ("tumor_truth", "organ_truth", "liver_probability", "tumor_probability"):
        assert item[key].shape[1:] == (256, 256)
    for key in ("liver_probability", "tumor_probability"):
        assert np.isfinite(item[key]).all()
        assert float(item[key].min()) >= 0 and float(item[key].max()) <= 1
    m1_cache[volume_id] = item
    coverage.append({"volume_id": volume_id, "slices": len(item["sample_id"]),
                     "tumor_positive_slices": int(item["tumor_truth"].any(axis=(1, 2)).sum()),
                     "cache_dtype": str(item["tumor_probability"].dtype),
                     "cache_mb": path.stat().st_size / (1024 ** 2)})
coverage_frame = pd.DataFrame(coverage)
assert len(m1_cache) == 13 and int(coverage_frame["slices"].sum()) == 10_685
coverage_frame.to_csv(OUT["mark_1"] / "cache_coverage.csv", index=False)
display(coverage_frame)
print("PASS: complete, ordered, finite validation cache.")

REUSE: copied 13 frozen patient caches from mark_1_outputs/probability_cache.


,volume_id,slices,tumor_positive_slices,cache_dtype,cache_mb
0,104,781,122,float16,83.453427
1,105,986,0,float16,105.489108
2,106,771,0,float16,81.110108
3,107,771,71,float16,81.570901
4,108,856,202,float16,91.343683
5,109,756,131,float16,80.252860
6,110,816,130,float16,85.879256
7,111,761,92,float16,80.794309
8,112,751,26,float16,79.941479
9,113,836,135,float16,88.831850


PASS: complete, ordered, finite validation cache.


### 1.4 Slice probability statistics and localization panels (V104, V116)

In [5]:
def probability_statistics(cached):
    rows = []
    for volume_id, item in cached.items():
        for index, sample_id in enumerate(item["sample_id"]):
            truth = item["tumor_truth"][index].astype(bool)
            organ = item["organ_truth"][index].astype(bool)
            probability = item["tumor_probability"][index].astype(np.float32)
            true_values = probability[truth]
            liver_background = probability[organ & ~truth]
            extra_liver = probability[~organ]
            rows.append({
                "sample_id": str(sample_id), "volume_id": volume_id,
                "slice_index": int(item["slice_index"][index]),
                "true_pixels": int(truth.sum()),
                "max_tumor_probability": float(probability.max()),
                "mean_probability_inside_truth": float(true_values.mean()) if true_values.size else np.nan,
                "max_probability_inside_truth": float(true_values.max()) if true_values.size else np.nan,
                "mean_liver_background_probability": float(liver_background.mean()) if liver_background.size else np.nan,
                "mean_extra_liver_probability": float(extra_liver.mean()) if extra_liver.size else np.nan,
            })
    return pd.DataFrame(rows)


def localization_panel(volume_id, statistics):
    item = m1_cache[volume_id]
    candidates = statistics.loc[
        statistics["volume_id"].eq(volume_id) & statistics["true_pixels"].gt(0)].copy()
    selected = list(dict.fromkeys([
        candidates["true_pixels"].idxmax(),
        (candidates["true_pixels"] - candidates["true_pixels"].median()).abs().idxmin(),
        candidates["true_pixels"].idxmin(),
        candidates["mean_probability_inside_truth"].idxmax(),
        candidates["mean_probability_inside_truth"].idxmin(),
    ]))
    figure, axes = plt.subplots(len(selected), 5, figsize=(18, 3.6 * len(selected)))
    if len(selected) == 1:
        axes = axes[None, :]
    for row_axes, row_index in zip(axes, selected):
        row = statistics.loc[row_index]
        index = int(np.where(item["slice_index"] == row["slice_index"])[0][0])
        manifest_row = validation_manifest.loc[
            validation_manifest["sample_id"].eq(row["sample_id"])].iloc[0]
        with Image.open(DATASET_ROOT / manifest_row["image_path"]) as handle:
            image = image_robust_normalize(
                np.asarray(handle.convert("L"), dtype=np.float32) / 255.0)
        truth = item["tumor_truth"][index].astype(bool)
        liver_probability = item["liver_probability"][index].astype(np.float32)
        tumor_probability = item["tumor_probability"][index].astype(np.float32)
        prediction = tumor_probability >= 0.50
        error = np.zeros((*truth.shape, 3), dtype=np.float32)
        error[truth & ~prediction, 0] = 1.0
        error[prediction & ~truth, 2] = 1.0
        panels = [
            (image, "Normalized CT", "gray", None),
            (image, "Truth contour", "gray", truth),
            (liver_probability, "Liver probability", "viridis", None),
            (tumor_probability, "Tumor probability", "magma", None),
            (error, "FN red / FP blue", None, None),
        ]
        for axis, (panel, title, cmap, contour) in zip(row_axes, panels):
            if panel.ndim == 2:
                axis.imshow(panel, cmap=cmap, vmin=0, vmax=1)
            else:
                axis.imshow(panel)
            if contour is not None and contour.any():
                axis.contour(contour, levels=[0.5], colors=["#00FFFF"], linewidths=1)
            axis.set_title(title)
            axis.axis("off")
        row_axes[0].set_ylabel(
            f"{row['sample_id']}\ntrue={row['true_pixels']:,}\npmax={row['max_tumor_probability']:.3f}",
            fontsize=8)
    figure.suptitle(f"Volume {volume_id} probability localization", fontsize=16, weight="bold")
    figure.tight_layout()
    path = OUT_FIGS["mark_1"] / f"probability_localization_{volume_id}.png"
    figure.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()


m1_slice_stats = probability_statistics(m1_cache)
m1_slice_stats.to_csv(OUT["mark_1"] / "probability_slice_statistics.csv", index=False)
for focus_volume in (104, 116):
    localization_panel(focus_volume, m1_slice_stats)
print("PASS: slice statistics + localization panels written.")

C:\Users\alanm\AppData\Local\Temp\ipykernel_20728\2595376239.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PASS: slice statistics + localization panels written.


C:\Users\alanm\AppData\Local\Temp\ipykernel_20728\2595376239.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.5 Fixed-bin probability populations

In [6]:
focus_volumes = [104, 116, 108, 109, 110]
bins = np.linspace(0, 1, 51)
figure, axes = plt.subplots(len(focus_volumes), 1, figsize=(12, 3.2 * len(focus_volumes)))
rng = np.random.default_rng(SEED)
for axis, volume_id in zip(axes, focus_volumes):
    item = m1_cache[volume_id]
    probability = item["tumor_probability"].astype(np.float32)
    truth = item["tumor_truth"].astype(bool)
    organ = item["organ_truth"].astype(bool)
    populations = {
        "True tumor": probability[truth],
        "Liver background": probability[organ & ~truth],
        "Extra-liver": probability[~organ],
    }
    for label, values in populations.items():
        if values.size > 500_000:
            values = rng.choice(values, 500_000, replace=False)
        axis.hist(values, bins=bins, density=True, histtype="step",
                  linewidth=1.5, label=f"{label} (n={len(values):,})")
    axis.set_yscale("log")
    axis.set_xlim(0, 1)
    axis.set_title(f"Volume {volume_id}")
    axis.set_xlabel("Tumor probability")
    axis.set_ylabel("Density (log)")
    axis.legend(fontsize=8)
figure.suptitle("Tumor-probability populations", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_1"] / "probability_population_histograms.png",
               dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_20728\2085684606.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.6 Source-NIfTI HU contrast (tumor vs liver)

In [7]:
import nibabel as nib


def robust_effect_size(tumor_values, liver_values):
    tumor_mad = np.median(np.abs(tumor_values - np.median(tumor_values)))
    liver_mad = np.median(np.abs(liver_values - np.median(liver_values)))
    pooled = max(1.4826 * np.sqrt((tumor_mad ** 2 + liver_mad ** 2) / 2), 1e-6)
    return float((np.median(tumor_values) - np.median(liver_values)) / pooled)


def compute_hu_contrast():
    rows = []
    positive_rows = validation_manifest.loc[validation_manifest["tumor_pixels"].gt(0)]
    for volume_id, group in positive_rows.groupby("volume_id", sort=True):
        first = group.iloc[0]
        ct_image = nib.load(str(first["source_volume_path"]))
        segmentation = nib.load(str(first["source_segmentation_path"]))
        transform = str(first["transform_applied"])
        for row in group.sort_values("slice_index").itertuples(index=False):
            z = int(row.slice_index)
            hu = np.asanyarray(ct_image.dataobj[:, :, z]).astype(np.float32)
            labels = np.asanyarray(segmentation.dataobj[:, :, z]).astype(np.uint8)
            if transform == "rot180":
                labels = np.rot90(labels, 2).copy()
            elif transform != "identity":
                raise ValueError(f"Unsupported transform: {transform}")
            tumor_values = hu[labels == 2]
            liver_values = hu[labels == 1]
            if not tumor_values.size or not liver_values.size:
                continue
            rows.append({
                "sample_id": row.sample_id, "volume_id": int(volume_id),
                "slice_index": z,
                "tumor_pixels_native": int(tumor_values.size),
                "liver_background_pixels_native": int(liver_values.size),
                "tumor_mean_hu": float(tumor_values.mean()),
                "tumor_median_hu": float(np.median(tumor_values)),
                "liver_background_mean_hu": float(liver_values.mean()),
                "liver_background_median_hu": float(np.median(liver_values)),
                "mean_contrast_hu": float(tumor_values.mean() - liver_values.mean()),
                "median_contrast_hu": float(np.median(tumor_values) - np.median(liver_values)),
                "robust_effect_size": robust_effect_size(tumor_values, liver_values),
            })
    return pd.DataFrame(rows)


hu_slice = compute_hu_contrast()
assert not hu_slice.empty
hu_slice.to_csv(OUT["mark_1"] / "hu_contrast_per_slice.csv", index=False)
hu_volume = hu_slice.groupby("volume_id").agg(
    positive_slices=("sample_id", "size"),
    median_contrast_hu=("median_contrast_hu", "median"),
    mean_contrast_hu=("mean_contrast_hu", "mean"),
    median_effect_size=("robust_effect_size", "median")).reset_index()
hu_volume.to_csv(OUT["mark_1"] / "hu_contrast_per_volume.csv", index=False)
display(hu_volume)

,volume_id,positive_slices,median_contrast_hu,mean_contrast_hu,median_effect_size
0,104,122,-89.5,-76.957218,-4.122858
1,107,71,-25.5,-22.661905,-1.082776
2,108,202,-44.0,-36.663048,-1.773188
3,109,131,-64.0,-49.882406,-2.457697
4,110,130,-34.0,-35.158978,-1.239959
5,111,92,-51.5,-52.125222,-1.774655
6,112,26,-94.0,-85.936717,-3.971641
7,113,135,-38.0,-38.439407,-1.704925
8,116,133,-10.0,-10.569320,-0.346739


### 1.7 Sweep global tumor thresholds and predicted-liver support

In [8]:
COARSE_TUMOR_THRESHOLDS = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
                                   dtype=np.float32)
LIVER_THRESHOLDS = np.array([0.30, 0.40, 0.50, 0.60, 0.70], dtype=np.float32)
LIVER_DILATION_KERNELS = [1, 5, 11, 21, 31]


def dilated_support(probability, threshold, kernel):
    binary = torch.from_numpy((probability >= threshold).astype(np.float32))[:, None]
    if kernel > 1:
        binary = F.max_pool2d(binary, kernel_size=kernel, stride=1, padding=kernel // 2)
    return binary[:, 0].numpy() > 0


def evaluate_configuration(tumor_threshold, liver_threshold=None, kernel=1):
    patient_rows, slice_rows = [], []
    total_intersection = total_predicted = total_true = 0
    positive_empty = positive_count = empty_fp = empty_count = 0
    removed_pixels = removed_true = removed_false = 0
    for volume_id, item in m1_cache.items():
        truth = item["tumor_truth"].astype(bool)
        raw = item["tumor_probability"].astype(np.float32) >= tumor_threshold
        if liver_threshold is None:
            prediction = raw
        else:
            support = dilated_support(item["liver_probability"].astype(np.float32),
                                      liver_threshold, kernel)
            prediction = raw & support
            removed = raw & ~prediction
            removed_pixels += int(removed.sum())
            removed_true += int((removed & truth).sum())
            removed_false += int((removed & ~truth).sum())
        intersection = prediction & truth
        predicted_pixels = prediction.sum(axis=(1, 2))
        true_pixels = truth.sum(axis=(1, 2))
        intersections = intersection.sum(axis=(1, 2))
        total_intersection += int(intersections.sum())
        total_predicted += int(predicted_pixels.sum())
        total_true += int(true_pixels.sum())
        positive = true_pixels > 0
        empty = ~positive
        positive_count += int(positive.sum())
        positive_empty += int((positive & (predicted_pixels == 0)).sum())
        empty_count += int(empty.sum())
        empty_fp += int((empty & (predicted_pixels > 0)).sum())
        patient_rows.append({
            "volume_id": volume_id, "true_pixels": int(true_pixels.sum()),
            "predicted_pixels": int(predicted_pixels.sum()),
            "intersection_pixels": int(intersections.sum()),
            "micro_dice": float((2 * intersections.sum() + 1e-6)
                                / (predicted_pixels.sum() + true_pixels.sum() + 1e-6)),
        })
        for index in range(len(true_pixels)):
            slice_rows.append({
                "sample_id": str(item["sample_id"][index]), "volume_id": volume_id,
                "true_pixels": int(true_pixels[index]),
                "predicted_pixels": int(predicted_pixels[index]),
                "intersection_pixels": int(intersections[index]),
            })
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = pd.qcut(
        positive_slices["true_pixels"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
    q1 = positive_slices.loc[positive_slices["size_quartile"].eq("Q1")]
    return {
        "tumor_threshold": float(tumor_threshold),
        "liver_threshold": float(liver_threshold) if liver_threshold is not None else np.nan,
        "dilation_kernel": int(kernel),
        "mode": "raw" if liver_threshold is None else "liver_supported",
        "global_dice": (2 * total_intersection + 1e-6) / (total_predicted + total_true + 1e-6),
        "pixel_precision": total_intersection / max(total_predicted, 1),
        "pixel_recall": total_intersection / max(total_true, 1),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_positive_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": 100 * float((q1["intersection_pixels"] > 0).mean()),
        "positive_predicted_empty_pct": 100 * positive_empty / max(positive_count, 1),
        "empty_slice_false_positive_pct": 100 * empty_fp / max(empty_count, 1),
        "predicted_tumor_pixels": total_predicted,
        "pixels_removed_by_liver_support": removed_pixels,
        "true_pixels_removed_by_liver_support": removed_true,
        "false_pixels_removed_by_liver_support": removed_false,
    }, patients


m1_config_rows, m1_patient_rows = [], []
for tumor_threshold in COARSE_TUMOR_THRESHOLDS:
    result, patients = evaluate_configuration(float(tumor_threshold))
    m1_config_rows.append(result)
    m1_patient_rows.append(patients.assign(tumor_threshold=float(tumor_threshold),
                                           liver_threshold=np.nan, dilation_kernel=1, mode="raw"))
    for liver_threshold in LIVER_THRESHOLDS:
        for kernel in LIVER_DILATION_KERNELS:
            result, patients = evaluate_configuration(float(tumor_threshold),
                                                      float(liver_threshold), kernel)
            m1_config_rows.append(result)
            m1_patient_rows.append(patients.assign(
                tumor_threshold=float(tumor_threshold), liver_threshold=float(liver_threshold),
                dilation_kernel=kernel, mode="liver_supported"))

m1_config_results = pd.DataFrame(m1_config_rows)
m1_patient_config = pd.concat(m1_patient_rows, ignore_index=True)
m1_config_results.to_csv(OUT["mark_1"] / "calibration_configuration_results.csv", index=False)
m1_patient_config.to_csv(OUT["mark_1"] / "calibration_patient_metrics.csv", index=False)
display(m1_config_results.sort_values(
    ["mean_patient_dice", "empty_slice_false_positive_pct"],
    ascending=[False, True]).head(15))
print(f"Evaluated {len(m1_config_results)} configurations (234 expected).")

,tumor_threshold,liver_threshold,dilation_kernel,mode,global_dice,pixel_precision,pixel_recall,mean_patient_dice,median_patient_dice,worst_positive_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,predicted_tumor_pixels,pixels_removed_by_liver_support,true_pixels_removed_by_liver_support,false_pixels_removed_by_liver_support
208,0.7,NaN,1,raw,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
209,0.7,0.3,1,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
210,0.7,0.3,5,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
211,0.7,0.3,11,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
212,0.7,0.3,21,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
213,0.7,0.3,31,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
214,0.7,0.4,1,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
215,0.7,0.4,5,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
216,0.7,0.4,11,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0
217,0.7,0.4,21,liver_supported,0.453658,0.727196,0.329656,0.332869,0.313549,6.526180e-12,1.443085e-11,6.526180e-12,27.376426,46.641075,3.194027,298962,0,0,0


Evaluated 234 configurations (234 expected).


### 1.8 Preregistered acceptance gate + calibration frontier

In [9]:
def gate_flags(frame):
    return (
        frame["mean_patient_dice"].ge(FINAL_TARGETS["mean_patient_dice"])
        & frame["volume_104_dice"].ge(FINAL_TARGETS["volume_104_dice"])
        & frame["volume_116_dice"].ge(FINAL_TARGETS["volume_116_dice"])
        & frame["q1_detected_pct"].ge(FINAL_TARGETS["q1_detected_pct"])
        & frame["positive_predicted_empty_pct"].le(FINAL_TARGETS["positive_predicted_empty_pct"])
        & frame["empty_slice_false_positive_pct"].le(FINAL_TARGETS["empty_slice_false_positive_pct"])
    )


m1_config_results["all_targets_passed"] = gate_flags(m1_config_results)
eligible = m1_config_results.loc[m1_config_results["all_targets_passed"]].copy()
if not eligible.empty:
    m1_selected_config = eligible.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]
else:
    m1_selected_config = m1_config_results.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]

figure, axes = plt.subplots(2, 3, figsize=(18, 10))
raw = m1_config_results.loc[m1_config_results["mode"].eq("raw")]
axes[0, 0].plot(raw["tumor_threshold"], raw["mean_patient_dice"], marker="o")
axes[0, 0].axhline(FINAL_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 0].set_title("Mean patient Dice")
axes[0, 1].plot(raw["tumor_threshold"], raw["volume_104_dice"], marker="o", label="V104")
axes[0, 1].plot(raw["tumor_threshold"], raw["volume_116_dice"], marker="s", label="V116")
axes[0, 1].set_title("Focus-patient Dice"); axes[0, 1].legend()
axes[0, 2].plot(raw["tumor_threshold"], raw["q1_detected_pct"], marker="o")
axes[0, 2].axhline(FINAL_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[0, 2].set_title("Q1 detection (%)")
axes[1, 0].plot(raw["empty_slice_false_positive_pct"], raw["mean_patient_dice"], marker="o")
axes[1, 0].axvline(FINAL_TARGETS["empty_slice_false_positive_pct"], linestyle="--", color="#444")
axes[1, 0].set_title("Patient Dice vs empty-slice FP"); axes[1, 0].set_xlabel("Empty-slice FP (%)")
axes[1, 1].plot(raw["positive_predicted_empty_pct"], raw["q1_detected_pct"], marker="o")
axes[1, 1].set_title("Q1 detection vs positive-empty")
axes[1, 1].set_xlabel("Positive predicted empty (%)")
removal = m1_config_results.loc[m1_config_results["mode"].eq("liver_supported")]
axes[1, 2].hist(removal["pixels_removed_by_liver_support"], bins=30, color="#2878B5")
axes[1, 2].set_title("Pixels removed by liver support")
for axis in axes.flat:
    axis.grid(alpha=0.25)
figure.suptitle("Mark 1 calibration frontier", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_1"] / "calibration_frontier_dashboard.png",
               dpi=170, bbox_inches="tight")
plt.show()
print("Selected diagnostic configuration:", m1_selected_config.to_dict())

Selected diagnostic configuration: {'tumor_threshold': 0.699999988079071, 'liver_threshold': nan, 'dilation_kernel': 1, 'mode': 'raw', 'global_dice': 0.4536579411116777, 'pixel_precision': 0.7271960985008128, 'pixel_recall': 0.32965623279913026, 'mean_patient_dice': 0.3328691349398703, 'median_patient_dice': 0.3135493949016889, 'worst_positive_patient_dice': 6.5261797701053565e-12, 'volume_104_dice': 1.4430847379149868e-11, 'volume_116_dice': 6.5261797701053565e-12, 'q1_detected_pct': 27.376425855513308, 'positive_predicted_empty_pct': 46.64107485604607, 'empty_slice_false_positive_pct': 3.1940267551591828, 'predicted_tumor_pixels': 298962, 'pixels_removed_by_liver_support': 0, 'true_pixels_removed_by_liver_support': 0, 'false_pixels_removed_by_liver_support': 0, 'all_targets_passed': False}


C:\Users\alanm\AppData\Local\Temp\ipykernel_20728\1214226524.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.9 Bootstrap patient uncertainty + write the Mark 1 gate

In [10]:
selected_filter = (
    m1_patient_config["mode"].eq(m1_selected_config["mode"])
    & m1_patient_config["tumor_threshold"].eq(m1_selected_config["tumor_threshold"])
    & m1_patient_config["dilation_kernel"].eq(m1_selected_config["dilation_kernel"])
)
if m1_selected_config["mode"] == "liver_supported":
    selected_filter &= m1_patient_config["liver_threshold"].eq(m1_selected_config["liver_threshold"])
selected_patients = m1_patient_config.loc[selected_filter].copy()
selected_positive = selected_patients.loc[
    selected_patients["true_pixels"].gt(0), "micro_dice"].to_numpy()
rng = np.random.default_rng(SEED)
bootstrap_means = np.array([
    rng.choice(selected_positive, size=len(selected_positive), replace=True).mean()
    for _ in range(1_000)
])
m1_bootstrap = pd.DataFrame([{
    "configuration_mode": m1_selected_config["mode"],
    "tumor_threshold": m1_selected_config["tumor_threshold"],
    "liver_threshold": m1_selected_config["liver_threshold"],
    "dilation_kernel": m1_selected_config["dilation_kernel"],
    "patients": len(selected_positive), "bootstrap_iterations": 1_000,
    "mean_dice_p2_5": np.percentile(bootstrap_means, 2.5),
    "mean_dice_p50": np.percentile(bootstrap_means, 50),
    "mean_dice_p97_5": np.percentile(bootstrap_means, 97.5),
    "original_median": np.median(selected_positive),
    "original_q25": np.percentile(selected_positive, 25),
    "original_q75": np.percentile(selected_positive, 75),
}])
m1_bootstrap.to_csv(OUT["mark_1"] / "bootstrap_confidence_intervals.csv", index=False)
display(m1_bootstrap)

calibration_passed = bool(m1_config_results["all_targets_passed"].any())
selected = m1_selected_config.to_dict()
if calibration_passed:
    failure_category, next_mark = "calibration_success", "freeze_global_configuration"
else:
    focus_stats = m1_slice_stats.loc[
        m1_slice_stats["volume_id"].isin([104, 116]) & m1_slice_stats["true_pixels"].gt(0)]
    localized_probability = float(focus_stats["max_probability_inside_truth"].median())
    if localized_probability >= 0.10:
        failure_category, next_mark = ("weak_but_localized",
                                       "stable_recall_objective_or_predicted_liver_normalization")
    elif hu_volume is not None and (
            hu_volume.loc[hu_volume["volume_id"].isin([104, 116]),
                          "median_effect_size"].abs().median() < 0.5):
        failure_category, next_mark = ("low_source_contrast",
                                       "controlled_multi_window_source_nifti_experiment")
    else:
        failure_category, next_mark = ("mislocalized_or_absent_signal",
                                       "predicted_liver_roi_or_capacity_experiment")

m1_gate = {
    "status": "mark_1_diagnostic_complete",
    "calibration_gate_passed": calibration_passed,
    "failure_category": failure_category,
    "selected_global_configuration": selected if calibration_passed else None,
    "best_observed_configuration_for_diagnosis": selected,
    "manifest_sha256": manifest_hash,
    "checkpoint_sha256": source_checkpoint_hash,
    "parent_checkpoint_sha256": checkpoint["source_checkpoint_sha256"],
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_images_accessed": False,
    "next_mark": next_mark,
    "manual_review_required": not calibration_passed,
}
(OUT["mark_1"] / "mark_1_gate_result.json").write_text(json.dumps(m1_gate, indent=2))
display(pd.DataFrame([m1_gate]).T.rename(columns={0: "value"}))

# ---- Reproduction check against the original gate ----
orig_m1 = json.loads((MARK1_DIR / "mark_1_outputs" / "mark_1_gate_result.json").read_text())
_check = orig_m1["best_observed_configuration_for_diagnosis"]
recomputed = m1_gate["best_observed_configuration_for_diagnosis"]
diffs = {k: abs(float(recomputed[k]) - float(_check[k])) for k in
         ["global_dice", "mean_patient_dice", "volume_104_dice", "volume_116_dice",
          "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"]}
print("Mark 1 reproduction check (|recomputed - original|):", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 1 gate drifted from the original!"
print("PASS: Mark 1 gate matches the original mark_1_gate_result.json.")

,configuration_mode,tumor_threshold,liver_threshold,dilation_kernel,patients,bootstrap_iterations,mean_dice_p2_5,mean_dice_p50,mean_dice_p97_5,original_median,original_q25,original_q75
0,raw,0.7,NaN,1,9,1000,0.134014,0.336343,0.511702,0.313549,0.070361,0.595275


,value
status,mark_1_diagnostic_complete
calibration_gate_passed,False
failure_category,mislocalized_or_absent_signal
selected_global_configuration,None
best_observed_configuration_for_diagnosis,"{'tumor_threshold': 0.699999988079071, 'liver_..."
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
checkpoint_sha256,9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c...
parent_checkpoint_sha256,5f18223c0f618d45d1433dc4a48d950fdea8a039db9ea5...
checkpoint_epoch,8
test_images_accessed,False


Mark 1 reproduction check (|recomputed - original|): {'global_dice': 0.0, 'mean_patient_dice': 0.0, 'volume_104_dice': 0.0, 'volume_116_dice': 0.0, 'q1_detected_pct': 0.0, 'positive_predicted_empty_pct': 0.0, 'empty_slice_false_positive_pct': 0.0}
PASS: Mark 1 gate matches the original mark_1_gate_result.json.


In [11]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_1")


PASS: mark_1_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\01_mark_1\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_20728\3332784003.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
